In [2]:
import pandas as pd
import sqlite3
import os

df = pd.read_excel("/run/media/bharathy/DATA/Sales_Dataset_2024.xlsx")

df.columns = df.columns.str.replace(" ", "_").str.replace("-", "_")

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDtypes:\n", df.dtypes)
df.head()

Shape: (2000, 10)

Columns: ['Date', 'Region', 'Product', 'Salesperson', 'Units_Sold', 'Unit_Price', 'Category', 'Revenue', 'Cost', 'Profit']

Dtypes:
 Date           datetime64[ns]
Region                 object
Product                object
Salesperson            object
Units_Sold            float64
Unit_Price            float64
Category               object
Revenue               float64
Cost                  float64
Profit                float64
dtype: object


,Date,Region,Product,Salesperson,Units_Sold,Unit_Price,Category,Revenue,Cost,Profit
0,2024-04-12,North,Smartwatch,Hannah,15.0,1224.0,Accessories,18360.0,16451.634258,1908.365742
1,2024-12-14,North,Monitor,Eva,5.0,1321.0,Office,6605.0,4457.351727,2147.648273
2,2024-09-27,North,Mobile,Bob,11.0,912.0,Electronics,10032.0,6563.644126,3468.355874
3,2024-04-16,West,Monitor,Charlie,18.0,325.0,Office,5850.0,4320.807092,1529.192908
4,2024-03-12,West,Headphones,Eva,13.0,1042.0,Accessories,13546.0,8270.122666,5275.877334


In [3]:
os.makedirs("../data", exist_ok=True)
conn = sqlite3.connect("../data/superstore.db")
df.to_sql("orders", conn, if_exists="replace", index=False)

# Verify it landed correctly
check = pd.read_sql("SELECT * FROM orders LIMIT 5", conn)
conn.close()
check

,Date,Region,Product,Salesperson,Units_Sold,Unit_Price,Category,Revenue,Cost,Profit
0,2024-04-12 00:00:00,North,Smartwatch,Hannah,15.0,1224.0,Accessories,18360.0,16451.634258,1908.365742
1,2024-12-14 00:00:00,North,Monitor,Eva,5.0,1321.0,Office,6605.0,4457.351727,2147.648273
2,2024-09-27 00:00:00,North,Mobile,Bob,11.0,912.0,Electronics,10032.0,6563.644126,3468.355874
3,2024-04-16 00:00:00,West,Monitor,Charlie,18.0,325.0,Office,5850.0,4320.807092,1529.192908
4,2024-03-12 00:00:00,West,Headphones,Eva,13.0,1042.0,Accessories,13546.0,8270.122666,5275.877334


In [5]:
pip install ollama

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [8]:
import ollama
import sqlite3
import pandas as pd

SCHEMA = """
Table: orders
Columns:
- Date (datetime) — order date
- Region (text) — North/South/East/West
- Product (text) — product name e.g. Smartwatch, Monitor, Mobile, Headphones
- Salesperson (text) — name of salesperson
- Units_Sold (float)
- Unit_Price (float)
- Category (text) — Accessories/Office/Electronics
- Revenue (float)
- Cost (float)
- Profit (float)
"""

def generate_sql(question: str) -> str:
    prompt = f"""You are a SQL expert. Given this table schema:

{SCHEMA}

Write a single valid SQLite SELECT query to answer this question:
"{question}"

Rules:
- Only output the raw SQL query, nothing else — no markdown, no explanation, no backticks
- Only use SELECT statements — never DROP, DELETE, UPDATE, INSERT, or ALTER
- Use the exact column and table names given above
"""
    response = ollama.chat(model="qwen2.5-coder:1.5b", messages=[
        {"role": "user", "content": prompt}
    ])
    return response['message']['content'].strip()


test_question = "Which region had the highest total revenue?"
sql = generate_sql(test_question)
print(sql)

```sql
SELECT Region
FROM orders
ORDER BY Revenue DESC
LIMIT 1;
```


In [9]:
#non mandatory Cleanup process for ppl with good hardware run the 8 gb variant of ollama

import re

def generate_sql(question: str) -> str:
    prompt = f"""You are a SQL expert. Given this table schema:

{SCHEMA}

Write a single valid SQLite SELECT query to answer this question:
"{question}"

Rules:
- Only output the raw SQL query, nothing else — no markdown, no explanation, no backticks
- Only use SELECT statements — never DROP, DELETE, UPDATE, INSERT, or ALTER
- Use the exact column and table names given above
"""
    response = ollama.chat(model="qwen2.5-coder:1.5b", messages=[
        {"role": "user", "content": prompt}
    ])
    raw = response['message']['content'].strip()

    # Strip markdown code fences if present
    match = re.search(r"```(?:sql)?\s*(.*?)```", raw, re.DOTALL)
    if match:
        raw = match.group(1).strip()

    return raw


test_question = "Which region had the highest total revenue?"
sql = generate_sql(test_question)
print(sql)

SELECT Region FROM orders ORDER BY Revenue DESC LIMIT 1;


In [11]:
def run_query(sql: str) -> pd.DataFrame:
    
    if not sql.strip().upper().startswith("SELECT"):
        raise ValueError("Only SELECT queries are allowed.")
    
    conn = sqlite3.connect("../data/superstore.db")
    result = pd.read_sql(sql, conn)
    conn.close()
    return result

question = "Which region had the lowest monthly revenue and which month"
sql = generate_sql(question)
print("Generated SQL:", sql)

result = run_query(sql)
result

Generated SQL: SELECT R.Region, M.Month
FROM (
    SELECT 
        Region,
        strftime('%Y-%m', Date) AS Month,
        SUM(Revenue) AS Total_Revenue
    FROM orders
    GROUP BY Region, strftime('%Y-%m', Date)
) AS Monthly_Revenue
ORDER BY Total_Revenue ASC
LIMIT 1


DatabaseError: Execution failed on sql 'SELECT R.Region, M.Month
FROM (
    SELECT 
        Region,
        strftime('%Y-%m', Date) AS Month,
        SUM(Revenue) AS Total_Revenue
    FROM orders
    GROUP BY Region, strftime('%Y-%m', Date)
) AS Monthly_Revenue
ORDER BY Total_Revenue ASC
LIMIT 1': no such column: R.Region

In [12]:
def generate_sql(question: str, error_context: str = None) -> str:
    error_note = f"\n\nYour previous attempt failed with this error: {error_context}\nFix the query." if error_context else ""
    
    prompt = f"""You are a SQL expert. Given this table schema:

{SCHEMA}

Write a single valid SQLite SELECT query to answer this question:
"{question}"

Rules:
- Only output the raw SQL query, nothing else — no markdown, no explanation, no backticks
- Only use SELECT statements — never DROP, DELETE, UPDATE, INSERT, or ALTER
- Use the exact column and table names given above
- Keep the query as simple as possible — avoid unnecessary subqueries or aliases unless needed{error_note}
"""
    response = ollama.chat(model="qwen2.5-coder:1.5b", messages=[
        {"role": "user", "content": prompt}
    ])
    raw = response['message']['content'].strip()
    match = re.search(r"```(?:sql)?\s*(.*?)```", raw, re.DOTALL)
    if match:
        raw = match.group(1).strip()
    return raw


def ask(question: str, max_retries: int = 2):
    error_context = None
    for attempt in range(max_retries + 1):
        sql = generate_sql(question, error_context)
        try:
            result = run_query(sql)
            return sql, result
        except Exception as e:
            error_context = str(e)
            print(f"Attempt {attempt+1} failed: {error_context}")
    raise RuntimeError(f"Failed after {max_retries+1} attempts. Last SQL:\n{sql}")

In [13]:
sql, result = ask("Which region had the lowest monthly revenue and which month?")
print("Final SQL:", sql)
result

Attempt 1 failed: Execution failed on sql 'SELECT 
    T1.Region, 
    T1.Month
FROM 
    (
        SELECT 
            DATE_PART('month', Date) AS Month, 
            Region, 
            SUM(Revenue) AS Monthly_Revenue
        FROM 
            orders
        GROUP BY 
            DATE_PART('month', Date), 
            Region
        ORDER BY 
            Monthly_Revenue ASC
    ) AS T1
ORDER BY 
    Monthly_Revenue ASC
LIMIT 1;': no such function: DATE_PART
Final SQL: SELECT 
    T1.Region, 
    T1.Month
FROM 
    (
        SELECT 
            strftime('%m', Date) AS Month, 
            Region, 
            SUM(Revenue) AS Monthly_Revenue
        FROM 
            orders
        GROUP BY 
            strftime('%m', Date), 
            Region
        ORDER BY 
            Monthly_Revenue ASC
    ) AS T1
ORDER BY 
    Monthly_Revenue ASC
LIMIT 1;


,Region,Month
0,Easst,03
